# 02 Load Public Dataset

This notebook downloads a public Parquet dataset, loads it with PySpark, performs basic data quality checks, writes a curated Parquet dataset and runs Spark SQL analytics.

In [6]:
from urllib.request import urlretrieve

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pipeline.config import (
    GREEN_TAXI_2024_01_URL,
    GREEN_TAXI_RAW_FILE,
    GREEN_TAXI_CURATED_OUTPUT,
    TRIPS_BY_PICKUP_HOUR_OUTPUT,
    PAYMENT_TYPE_SUMMARY_OUTPUT,
    ensure_data_directories,
)

from pipeline.transformations import (
    select_green_taxi_columns,
    clean_green_taxi_trips,
)

from pipeline.analytics import (
    trips_by_pickup_hour,
    payment_type_summary,
)

In [7]:
ensure_data_directories()

raw_file = GREEN_TAXI_RAW_FILE
curated_output = GREEN_TAXI_CURATED_OUTPUT

In [8]:
spark = (
    SparkSession.builder
    .appName("Public Parquet Dataset Pipeline")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

spark.version

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 17:09:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

In [9]:
if not raw_file.exists():
    print(f"Downloading dataset to {raw_file} ...")
    urlretrieve(GREEN_TAXI_2024_01_URL, raw_file)
    print("Download finished.")
else:
    print(f"Dataset already exists: {raw_file}")

Dataset already exists: /workspace/data/raw/green_tripdata_2024-01.parquet


In [10]:
df_raw = spark.read.parquet(str(raw_file))

df_raw.printSchema()

df_raw.show(5, truncate=False)

df_raw.count()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

+--------+--------------------+---------------------+------------------+----------+-----

56551

In [11]:
df_selected = select_green_taxi_columns(df_raw)

df_selected.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+----------+------------+------------+---------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|trip_type|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+----------+------------+------------+---------+
|       2| 2024-01-01 00:46:55|  2024-01-01 00:58:25|                 N|         1|         236|         239|              1|         1.98|       12.8|      3.61|       21.66|           1|        1|
|       2| 2024-01-01 00:31:42|  2024-01-01 00:52:34|                 N|         1|          65|         170|              5|         6.54|       30.3|      7.11|       42.66|           1|        1|
|    

## Basic Data Quality Checks

Before transforming the dataset, we inspect basic summary statistics for key numerical columns. This helps identify invalid values, missing data and outliers, such as negative fares or unrealistic trip distances.

In [12]:
df_selected.describe(
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount"
).show()

df_selected.select(
    F.count("*").alias("row_count"),
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("null_passenger_count"),
    F.sum(F.col("trip_distance").isNull().cast("int")).alias("null_trip_distance"),
    F.sum(F.col("fare_amount").isNull().cast("int")).alias("null_fare_amount"),
    F.sum(F.col("total_amount").isNull().cast("int")).alias("null_total_amount"),
).show()

+-------+------------------+------------------+------------------+------------------+------------------+
|summary|   passenger_count|     trip_distance|       fare_amount|        tip_amount|      total_amount|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|             53136|             56551|             56551|             56551|             56551|
|   mean|1.3091689250225835|31.491123941221307|16.929275167547825|2.2565100528727724|22.403186327385683|
| stddev|0.9782520125657749|1417.4603816839042|15.356031505227904|2.8479566702861248|16.956517665111612|
|    min|                 0|               0.0|             -70.0|             -1.66|             -76.5|
|    max|                 9|         201421.68|            1422.6|             110.0|            1424.1|
+-------+------------------+------------------+------------------+------------------+------------------+

+---------+--------------------+------------------+---

The summary statistics show that the raw dataset contains invalid or implausible records, including zero-distance trips, negative fare amounts and extreme distance outliers. These records are filtered in the cleaning step below.

In [13]:
df_clean = clean_green_taxi_trips(df_selected)

df_clean.select(
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tip_percentage",
    "total_amount",
    "pickup_date",
    "pickup_hour",
).show(10)

+--------------------+---------------------+---------------------+-------------+-----------+----------+--------------+------------+-----------+-----------+
|lpep_pickup_datetime|lpep_dropoff_datetime|trip_duration_minutes|trip_distance|fare_amount|tip_amount|tip_percentage|total_amount|pickup_date|pickup_hour|
+--------------------+---------------------+---------------------+-------------+-----------+----------+--------------+------------+-----------+-----------+
| 2024-01-01 00:46:55|  2024-01-01 00:58:25|                 11.5|         1.98|       12.8|      3.61|          28.2|       21.66| 2024-01-01|          0|
| 2024-01-01 00:31:42|  2024-01-01 00:52:34|   20.866666666666667|         6.54|       30.3|      7.11|         23.47|       42.66| 2024-01-01|          0|
| 2024-01-01 00:30:21|  2024-01-01 00:49:23|   19.033333333333335|         3.08|       19.8|       3.0|         15.15|       28.05| 2024-01-01|          0|
| 2024-01-01 00:30:20|  2024-01-01 00:42:12|   11.86666666666666

In [14]:
raw_count = df_selected.count()
clean_count = df_clean.count()

print(f"Raw rows:     {raw_count}")
print(f"Clean rows:   {clean_count}")
print(f"Removed rows: {raw_count - clean_count}")
print(f"Kept share:   {clean_count / raw_count:.2%}")

Raw rows:     56551
Clean rows:   52781
Removed rows: 3770
Kept share:   93.33%


In [15]:
df_clean.write.mode("overwrite").parquet(str(curated_output))

df_curated = spark.read.parquet(str(curated_output))

df_curated.printSchema()
df_curated.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- tip_percentage: double (nullable = true)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+----------+--------

In [16]:
df_curated.createOrReplaceTempView("green_taxi_trips")

spark.sql("""
    SELECT
        pickup_hour,
        COUNT(*) AS trip_count,
        ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
        ROUND(AVG(total_amount), 2) AS avg_total_amount,
        ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
    FROM green_taxi_trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""").show(24)

spark.sql("""
    SELECT
        payment_type,
        COUNT(*) AS trip_count,
        ROUND(AVG(total_amount), 2) AS avg_total_amount,
        ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
    FROM green_taxi_trips
    GROUP BY payment_type
    ORDER BY trip_count DESC
""").show()

+-----------+----------+-----------------+----------------+------------------+
|pickup_hour|trip_count|avg_trip_distance|avg_total_amount|avg_tip_percentage|
+-----------+----------+-----------------+----------------+------------------+
|          0|       926|             2.96|           20.86|             12.88|
|          1|       687|             2.95|           21.31|             11.96|
|          2|       471|             2.91|            22.0|             12.15|
|          3|       353|             3.59|           25.64|             10.98|
|          4|       278|             4.15|           26.41|               9.2|
|          5|       251|           143.73|           30.08|             13.24|
|          6|       751|             2.93|           20.37|             15.69|
|          7|      2094|            63.41|           20.75|             14.55|
|          8|      2595|            42.76|           21.97|             14.65|
|          9|      2794|           112.55|          

In [17]:
df_trips_by_hour = trips_by_pickup_hour(df_curated)
df_payment_summary = payment_type_summary(df_curated)

df_trips_by_hour.show(24)
df_payment_summary.show()

+-----------+----------+-----------------+----------------+------------------+
|pickup_hour|trip_count|avg_trip_distance|avg_total_amount|avg_tip_percentage|
+-----------+----------+-----------------+----------------+------------------+
|          0|       926|             2.96|           20.86|             12.88|
|          1|       687|             2.95|           21.31|             11.96|
|          2|       471|             2.91|            22.0|             12.15|
|          3|       353|             3.59|           25.64|             10.98|
|          4|       278|             4.15|           26.41|               9.2|
|          5|       251|           143.73|           30.08|             13.24|
|          6|       751|             2.93|           20.37|             15.69|
|          7|      2094|            63.41|           20.75|             14.55|
|          8|      2595|            42.76|           21.97|             14.65|
|          9|      2794|           112.55|          

In [18]:
df_trips_by_hour.write.mode("overwrite").parquet(
    str(TRIPS_BY_PICKUP_HOUR_OUTPUT)
)

df_payment_summary.write.mode("overwrite").parquet(
    str(PAYMENT_TYPE_SUMMARY_OUTPUT)
)